In [ ]:
# Cell 1: get the binary (pinned to commit 539f245; may not support the newest model archs)
import pathlib, tarfile, os, subprocess, time, requests
from huggingface_hub import hf_hub_download

p = pathlib.Path(hf_hub_download("afsafwtggg/llama-cpp-t4-cache", "llama-cpp-t4-539f245.tar.gz"))
with tarfile.open(p) as f: f.extractall("/kaggle/working/engine")
BIN = pathlib.Path("/kaggle/working/engine/llama-cpp-t4-539f245/bin")
for b in BIN.iterdir():
    if b.is_file(): b.chmod(0o755)

llama-cpp-t4-539f245.tar.gz: reconstructing file:   0%|          |  0.00B / 94.2MB            

llama-cpp-t4-539f245.tar.gz: downloading bytes:           |  0.00B            

/tmp/ipykernel_50/3223120646.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(p) as f: f.extractall("/kaggle/working/engine")


In [ ]:
# Cell 2: get a GGUF (fill in your own repo/file)
MODEL = hf_hub_download("unsloth/Qwen3.6-35B-A3B-MTP-GGUF", "Qwen3.6-35B-A3B-UD-Q3_K_XL.gguf", local_dir="/kaggle/working/models")

Qwen3.6-35B-A3B-UD-Q3_K_XL.gguf: reconstructing file:   0%|          |  0.00B / 17.2GB            

Qwen3.6-35B-A3B-UD-Q3_K_XL.gguf: downloading bytes:           |  0.00B            

In [ ]:
# Cell 4 (optional): public URL via ngrok
!pip -q install pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("<ngrok auth token>")   # use Kaggle Secrets, don't hardcode
print(ngrok.connect(8080).public_url)

In [ ]:
import glob, os, subprocess, time, requests

nv = sorted(glob.glob("/usr/local/lib/python3.13/dist-packages/nvidia/*/lib"))
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = ":".join([str(BIN), *nv, env.get("LD_LIBRARY_PATH", "")])

# verify before launching
print(os.path.exists(f"{BIN}/libllama-server-impl.so"))
r = subprocess.run(["ldd", f"{BIN}/llama-server"], env=env, capture_output=True, text=True)
missing = [l for l in r.stdout.splitlines() if "not found" in l]
print("missing:", missing or "none")

if not missing:
    proc = subprocess.Popen(
    [str(BIN/"llama-server"), "-m", MODEL, "-ngl", "99",
     "-c", "262144", "--jinja", "-fa", "on",
     "-ctk", "q8_0", "-ctv", "q8_0",
     "-ts", "1,1",
     "--alias", "claude-sonnet-4-5",
     "--host", "0.0.0.0", "--port", "8080", "--api-key", "CHANGE_ME"],
    env=env, stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(300):
        if proc.poll() is not None:
            print("server died, exit", proc.returncode)
            print(open("/kaggle/working/server.log").read()[-3000:])
            break
        try:
            if requests.get("http://127.0.0.1:8080/health").json().get("status") == "ok":
                print("ready"); break
        except Exception: pass
        time.sleep(2)

True
missing: none
ready


In [ ]:
!ps aux | grep llama-server | grep -v grep

root         440 11.2  6.6 103308640 2180672 ?   Sl   13:52   0:09 /kaggle/working/engine/llama-cpp-t4-539f245/bin/llama-server -m /kaggle/working/models/Qwen3.6-35B-A3B-UD-Q3_K_XL.gguf -ngl 99 -c 262144 --jinja -fa on -ctk q8_0 -ctv q8_0 -ts 1,1 --alias claude-sonnet-4-5 --host 0.0.0.0 --port 8080 --api-key CHANGE_ME
